# mBART Fine-Tuning: Car Rental Review Summarization

Fine-tunes `facebook/mbart-large-cc25` on German/French → English summarization
using LoRA adapters.

mBART is a multilingual encoder-decoder model pre-trained on 25 languages.
It is much better than flan-t5 for cross-lingual generation tasks.

This notebook handles **summarization only**.
Categories and sentiment are handled by the flan-t5 notebook.

> **Tip:** Go to `Runtime → Change runtime type` and select **GPU** (T4) for faster training.

In [ ]:
!pip install -q -U torchao transformers datasets peft accelerate huggingface_hub sentencepiece

In [ ]:
import time
import torch
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    MBartForConditionalGeneration,
    MBart50TokenizerFast,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

## 0. Config

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME   = "facebook/mbart-large-cc25"
ADAPTER_PATH = "miladghofrani/car-rental-mbart-summarizer"

# ── Dataset ────────────────────────────────────────────────────────────────
HF_DATASET = "miladghofrani/car-rental-reviews-short"

# ── Training ───────────────────────────────────────────────────────────────
MAX_STEPS     = None   # None = full training run
NUM_EPOCHS    = 5
LEARNING_RATE = 5e-4

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_RANK    = 32     # mBART-large is 610M params — fits on T4 with rank 32
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# ── mBART language codes ───────────────────────────────────────────────────
# Reviews are German or French; summaries are always English
SRC_LANG = "de_DE"   # dominant language in dataset (87% German)
TGT_LANG = "en_XX"   # output is always English

## 1. Detect Device

In [ ]:
def get_device():
    print('🚀 Initializing...')
    print(f'✅ PyTorch version: {torch.__version__}')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'✅ Device: {device.upper()}')
    return device

device = get_device()

## 2. Load Dataset

Loads `miladghofrani/car-rental-reviews` and keeps **only summarization examples**
(review body → English summary). Categories and sentiment are not used here.

In [ ]:
def load_summarization_dataset():
    print(f"\n📥 Loading dataset from HF Hub ({HF_DATASET})...")
    raw = load_dataset(HF_DATASET, split="train")
    print(f"✅ Loaded {raw.num_rows:,} reviews")

    inputs, outputs = [], []
    for row in raw:
        body    = (row.get("review_body") or "").strip()
        summary = (row.get("summary") or "").strip()
        if not body or not summary:
            continue
        inputs.append(body)
        outputs.append(summary)

    split = Dataset.from_dict({"input": inputs, "output": outputs}).train_test_split(
        test_size=0.1, seed=42
    )
    print(f"✅ {split['train'].num_rows:,} train / {split['test'].num_rows:,} validation examples")

    print("\n🔍 Sample:")
    print(f"  Input  : {split['train'][0]['input'][:200]}...")
    print(f"  Output : {split['train'][0]['output']}")
    return split

dataset = load_summarization_dataset()

assert dataset["train"].num_rows > 3000, "Dataset too small — check internet/dataset!"

import shutil, os
shutil.rmtree(os.path.expanduser("~/.cache/huggingface/datasets"), ignore_errors=True)
print("\nCache cleared.")

## 3. Load Model & Tokenizer

mBART uses a specialized tokenizer that requires setting source and target
language codes before tokenizing inputs and labels.

In [ ]:
def print_number_of_trainable_model_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(
        f"📊 Model Parameter Report:\n"
        f"--------------------------------------------------\n"
        f"  Total Parameters:     {total:,}\n"
        f"  Trainable Parameters: {trainable:,}\n"
        f"  Percentage Trainable: {100 * trainable / total:.4f}%\n"
        f"--------------------------------------------------"
    )

print("\n🔤 Loading Tokenizer...")
tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
tokenizer.src_lang = SRC_LANG
print(f"✅ Tokenizer loaded. src_lang={SRC_LANG}, tgt_lang={TGT_LANG}")

print("\n🧠 Loading mBART model...")
model = MBartForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
print(f"✅ {MODEL_NAME} loaded.")
print_number_of_trainable_model_parameters(model)

## 4. Tokenize Dataset

In [ ]:
def tokenize_dataset(tokenizer, dataset):
    print("\n⚙️  Tokenizing dataset...")
    pad_id = tokenizer.pad_token_id

    def tokenize(batch):
        tokenizer.src_lang = SRC_LANG
        tokenizer.tgt_lang = TGT_LANG

        model_inputs = tokenizer(
            batch["input"], truncation=True, max_length=512
        )
        labels = tokenizer(
            text_target=batch["output"], truncation=True, max_length=128
        )
        # Replace padding token id in labels with -100 so loss ignores them
        model_inputs["labels"] = [
            [(t if t != pad_id else -100) for t in label]
            for label in labels["input_ids"]
        ]
        return model_inputs

    tokenized = dataset.map(tokenize, batched=True, remove_columns=["input", "output"])
    print("✅ Tokenization complete.")
    return tokenized

tokenized_dataset = tokenize_dataset(tokenizer, dataset)

## 5. Inject LoRA Adapters

In [ ]:
print("\n🪄 Injecting LoRA Adapters (PEFT)...")
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    # mBART attention projection layer names
    target_modules=["q_proj", "v_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)
peft_model = get_peft_model(model, lora_config)
print("✅ LoRA adapters injected.")
print_number_of_trainable_model_parameters(peft_model)

## 6. Train & Save

In [ ]:
run_dir = f"./mbart-summarizer-run-{int(time.time())}"

training_args = TrainingArguments(
    output_dir=run_dir,
    auto_find_batch_size=True,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    logging_steps=1,
    max_steps=MAX_STEPS if MAX_STEPS is not None else -1,
    gradient_checkpointing=True,
    fp16=True,
)

collator = DataCollatorForSeq2Seq(tokenizer, model=peft_model, padding=True)

# Required when using gradient checkpointing with PEFT
peft_model.enable_input_require_grads()

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=collator,
)

mode = f"max_steps={MAX_STEPS}" if MAX_STEPS is not None else f"{NUM_EPOCHS} full epoch(s)"
print(f"\n🏋️  Starting mBART summarization training ({mode})...")
trainer.train()

trainer.model.save_pretrained("./mbart-checkpoint-local")
tokenizer.save_pretrained("./mbart-checkpoint-local")
print("\n✅ Training complete! Adapter saved to ./mbart-checkpoint-local")

## 7. Push to HuggingFace Hub

In [ ]:
from huggingface_hub import login
import os

hf_token = ""
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN", "")

login(token=hf_token)

peft_model.push_to_hub(ADAPTER_PATH)
tokenizer.push_to_hub(ADAPTER_PATH)
print(f"\n✅ Adapter pushed to: https://huggingface.co/{ADAPTER_PATH}")

## 8. Test Inference

Quick smoke test — run a single German review through the trained model.

In [ ]:
test_review = (
    "Katastrophale Abwicklung. Insgesamt standen wir über eine Stunde an. "
    "Es wird versucht mit Druck Zusatzprodukte zu verkaufen - Versicherungen, anderer Wagen etc. "
    "Bei Rückgabe hat man uns zwei Schäden angelastet, die definitiv bereits vorhanden waren."
)

tokenizer.src_lang = SRC_LANG
inputs = tokenizer(test_review, return_tensors="pt", truncation=True, max_length=512).to(device)

peft_model.eval()
with torch.no_grad():
    output_tokens = peft_model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id[TGT_LANG],
        max_new_tokens=80,
        num_beams=4,
    )

summary = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
print("Input  :", test_review)
print("Summary:", summary)